# Data Preprocessing & Feature Engineering

This notebook prepares the Telco Customer Churn dataset for machine learning.

The main steps include handling missing values, defining the target variable, reviewing potential data leakage, and preparing numerical and categorical features for model training.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path


#load the original dataset
file_path=Path.cwd().parent / "data" / "raw" / "Telco_customer_churn.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name = "Telco_Churn"
)

#preserve the original data
data = df.copy()

print("Dataset shape:", data.shape)
display(data.head())

Dataset shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 3. Data Leakage Analysis

Data leakage occurs when a machine learning model uses information that would not be available at the time of prediction.

In this section, we identify features that may reveal the target variable or contain information generated after customer churn.

The goal is to ensure that the model learns from information genuinely available before churn occurs.

In [9]:
columns_to_check = [
    "Churn Score",
    "CLTV",
    "Churn Value",
    "Churn Label",
    "Churn Reason"
]

display(data[columns_to_check].head(10))

,Churn Score,CLTV,Churn Value,Churn Label,Churn Reason
0,86,3239,1,Yes,Competitor made better offer
1,67,2701,1,Yes,Moved
2,86,5372,1,Yes,Moved
3,84,5003,1,Yes,Moved
4,89,5340,1,Yes,Competitor had better devices
5,78,5925,1,Yes,Competitor offered higher download speeds
6,100,5433,1,Yes,Competitor offered more data
7,92,4832,1,Yes,Competitor made better offer
8,77,5789,1,Yes,Competitor had better devices
9,97,2915,1,Yes,Competitor had better devices


### Data Leakage Considerations

Several columns require special attention before model training.

- Customer identifiers and duplicate target representations will not be used as predictive features.
- Churn Reason contains post-outcome information and will be excluded.
- Churn Score and CLTV require further investigation because their availability at prediction time is uncertain.

These decisions help prevent information leakage and improve the reliability of model evaluation.

## 4. Target Variable and Feature Selection

The target variable is Churn Value.

All features that directly reveal the churn outcome, represent customer identifiers, or may introduce data leakage are excluded.

Churn Score and CLTV are also excluded from the initial model because their availability at prediction time is uncertain.

In [10]:
# Target variable
y = data["Churn Value"]

# Columns excluded from model features
columns_to_drop = [
    "CustomerID",
    "Churn Label",
    "Churn Value",
    "Churn Reason",
    "Churn Score",
    "CLTV",
    "Count",
    "Country",
    "State"
]

# Feature matrix
X = data.drop(columns=columns_to_drop)

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (7043, 24)
Target vector shape: (7043,)

Target distribution:
Churn Value
0    5174
1    1869
Name: count, dtype: int64


### Feature Selection

The following columns were excluded from the initial modeling dataset:

- CustomerID: Customer identifier.
- Churn Label: Duplicate representation of the target.
- Churn Reason: Contains post-churn information.
- Churn Score: Potentially precomputed churn prediction.
- CLTV: Prediction-time availability is uncertain.
- Count, Country, State: Constant or non-informative features.
- Churn Value: Separated as the target variable.

The remaining features are used as the initial input for model development.

## 5. Numerical and Categorical Feature Separation

In this section, we separate numerical and categorical features.

Numerical features may require scaling, while categorical features need encoding before being used by machine learning algorithms.

This separation allows us to apply different preprocessing strategies to each feature type.

In [11]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical Features:", len(numerical_features))
print(numerical_features)

print("\nCategorical Features:", len(categorical_features))
print(categorical_features)

Numerical Features: 5
['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges']

Categorical Features: 19
['City', 'Lat Long', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Total Charges']


C:\Users\Adil Abdullayev\AppData\Local\Temp\ipykernel_10488\1832250132.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


## 6. Data Type Correction

Total Charges represents a numerical amount but is currently stored as a categorical or string variable.

We convert it to a numeric format and handle missing values during preprocessing.

In [12]:
# Convert Total Charges to numeric
X["Total Charges"] = pd.to_numeric(
    X["Total Charges"].astype(str).str.strip(),
    errors="coerce"
)

# Re-identify numerical and categorical features
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool", "str"]
).columns.tolist()

print("Numerical Features:", len(numerical_features))
print(numerical_features)

print("\nCategorical Features:", len(categorical_features))
print(categorical_features)

print("\nMissing Total Charges:", X["Total Charges"].isna().sum())

Numerical Features: 6
['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges']

Categorical Features: 18
['City', 'Lat Long', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Missing Total Charges: 11


### Data Type Correction

The Total Charges column was converted into a numerical data type because it represents monetary values.

The 11 missing values were preserved for the preprocessing stage rather than being removed from the dataset.

Numerical and categorical features were identified separately to support different preprocessing transformations.

## 7. Train-Test Split

We split the dataset into training and testing sets before applying preprocessing transformations.

The training set is used to fit the preprocessing steps and train machine learning models.

The test set is kept separate to evaluate model performance on unseen data.

Stratified sampling is used to preserve the original churn distribution in both sets.

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

Training features: (5634, 24)
Testing features: (1409, 24)

Training target distribution:
Churn Value
0    0.735
1    0.265
Name: proportion, dtype: float64

Testing target distribution:
Churn Value
0    0.735
1    0.265
Name: proportion, dtype: float64


### Train-Test Split

The dataset was divided into training and testing sets using an 80/20 split.

Stratified sampling was applied to preserve the churn distribution in both sets.

A fixed random state was used to ensure reproducibility.

Preprocessing transformations will be fitted using training data only.

## 8. Preprocessing Pipeline

We use a ColumnTransformer to apply different preprocessing steps to numerical and categorical features.

Numerical features:
- Missing values are imputed using the median.
- Features are standardized using StandardScaler.

Categorical features:
- Missing values are imputed using the most frequent value.
- Features are transformed using One-Hot Encoding.

The preprocessing pipeline is fitted on the training data only to prevent data leakage.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Numerical preprocessing
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data without fitting again
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (5634, 2827)
Processed testing shape: (1409, 2827)


### Preprocessing Pipeline

A ColumnTransformer was implemented to handle numerical and categorical features separately.

Numerical features were imputed using the median and standardized with StandardScaler.

Categorical features were imputed using the most frequent value and encoded with One-Hot Encoding.

The preprocessing pipeline was fitted exclusively on training data and then applied to the test data to prevent data leakage.

## 9. Preprocessing Validation

We validate the processed datasets before model training.

The training and testing datasets must have the same number of features, and missing values should have been handled by the preprocessing pipeline.

In [16]:
from scipy import sparse

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

if sparse.issparse(X_train_processed):
    print("Training missing values:", np.isnan(X_train_processed.data).sum())
    print("Testing missing values:", np.isnan(X_test_processed.data).sum())
else:
    print("Training missing values:", np.isnan(X_train_processed).sum())
    print("Testing missing values:", np.isnan(X_test_processed).sum())

print(
    "Same number of features:",
    X_train_processed.shape[1] == X_test_processed.shape[1]
)

Training shape: (5634, 2827)
Testing shape: (1409, 2827)
Training missing values: 0
Testing missing values: 0
Same number of features: True


### Preprocessing Validation

The preprocessing pipeline successfully transformed the training and testing datasets.

One-Hot Encoding expanded the original 24 features into 2,827 processed features.

Both datasets contain the same number of transformed features, allowing consistent model training and evaluation.

## 10. Preprocessing Summary

The preprocessing stage was completed successfully.

Key steps:

- Selected 24 initial features and separated the target variable.
- Excluded identifiers, duplicate target information, post-churn information, and potentially leakage-prone features.
- Converted Total Charges into a numerical format.
- Split the dataset into training (80%) and testing (20%) sets using stratified sampling.
- Applied median imputation and standard scaling to numerical features.
- Applied most-frequent imputation and One-Hot Encoding to categorical features.
- Fitted preprocessing transformations on training data only.

The processed datasets are now ready for machine learning model training.